# Initialization

In [ ]:
Dropbox_path="/mnt/c/Users/ETTINA03/NYU Langone Health Dropbox/April Jauhal/Shenhav_Lab"

In [2]:
import sys
import os
import re
import pickle
import random
import numpy as np
import pandas as pd
import sklearn.tree as tree
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.model_selection import (train_test_split, LeaveOneOut)
from sklearn import metrics
from sklearn.metrics import (roc_curve, auc, confusion_matrix, f1_score)
from collections import Counter
import shap
import matplotlib.pyplot as plt
import xgboost as xgb
import seaborn as sns

import scipy

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest
import statsmodels.api as sm
import re

from venny4py.venny4py import *

# prevent SettingWithCopy warning from train_test_split
pd.options.mode.chained_assignment = None

/home/april/miniforge3/envs/py-3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
%run ml_functions.py

In [4]:
seedlist1=[456, 1234, 567, 299, 101, 203, 678, 953, 578, 344]
seedlist2=[777,432, 555, 676, 921, 469, 903, 158, 851, 700]

# Data preprocessing and loading

### Loading Untargeted Data

In [ ]:
# Use full path as string not variable
%run load_untargeted_data.py -d "/mnt/c/Users/ETTINA03/NYU Langone Health Dropbox/April Jauhal/Shenhav_Lab" # Enter full path to "Dropbox_path" here

loading metadata
loading data
MISAME untargeted missing values: 0.01863350737978256
(843, 37072)
Mumta-LW untargeted missing values: 0.04413438269975361
(300, 60609)
CHILD untargeted missing values: 0.04528492111153207
(393, 60609)
reading keys
reading Sapient top metabolites


In [6]:
vital_bottom_limit = -2
vital_top_limit = -1
misame_bottom_limit = -1
misame_top_limit = 0
child_bottom_limit = -1
child_top_limit = 0

vital_meta_for_ML['WAZ_M04_wide_margin'] = vital_meta_for_ML['WAZ_M04'].apply(lambda x: 1 if x<vital_bottom_limit else (0 if x>vital_top_limit else np.nan))
misame_meta_for_ML['WAZ_M04_wide_margin'] = misame_meta_for_ML['WAZ_M04'].apply(lambda x: 1 if x<misame_bottom_limit else (0 if x>misame_top_limit else np.nan))
child_meta_for_ML['WAZ_M03_wide_margin'] = child_meta_for_ML['WAZ_M03'].apply(lambda x: 1 if x<child_bottom_limit else (0 if x>child_top_limit else np.nan))

In [7]:
misame_full_BM_df_filt = misame_full_BM_df_filt.rename(columns={'BEP': 'postnatal_BEP'})
vital_full_BM_df_filt = vital_full_BM_df_filt.rename(columns={'BEP': 'postnatal_BEP'})

### Loading Targeted Data

In [8]:
# Function to remove index suffix from child data
def remove_suffix(index_value):
    return re.sub(r'-\d+$', '', index_value)

# Function for macronutrient conversions
def percent_to_kcalL(col, nut): # convert from "percent" (g/100mL) to kcal/L, nut can be protein, fat, or carb
    if nut in ['protein', 'carb']:
        kcals_per_gram = 4
    elif nut == "fat":
        kcals_per_gram = 9
    col = col*10 # convert g/100ml to g/L
    col = col*kcals_per_gram # convert g/L to kcal/L
    return(col)

In [9]:
vital_macronut=pd.read_csv(os.path.join(Dropbox_path, "IMiC/Data/Targeted_metabolomics/VITAL/Allen_Lab/Allen_Macronutrients_VITAL.csv"), sep=',', index_col=0)
vital_macronut['Protein'] = percent_to_kcalL(vital_macronut['Protein'], 'protein')
vital_macronut['Fat'] = percent_to_kcalL(vital_macronut['Fat'], 'fat')
vital_macronut['CHO'] = percent_to_kcalL(vital_macronut['CHO'], 'carb')
misame_macronut=pd.read_csv(os.path.join(Dropbox_path, "IMiC/Data/Targeted_metabolomics/MISAME/Allen_Lab/Macronutrient_MISAME(in).csv"), sep=',', index_col=0)
misame_macronut['PROTEIN'] = percent_to_kcalL(misame_macronut['PROTEIN'], 'protein')
misame_macronut['FAT'] = percent_to_kcalL(misame_macronut['FAT'], 'fat')
misame_macronut['CARBOHYDRATE'] = percent_to_kcalL(misame_macronut['CARBOHYDRATE'], 'carb')
child_macronut=pd.read_csv(os.path.join(Dropbox_path, "IMiC/Data/Targeted_metabolomics/CHILD/Allen_Lab/Macronutrient_CHILD.csv"), sep=',', index_col=0)
child_macronut['PROTEIN'] = percent_to_kcalL(child_macronut['PROTEIN'], 'protein')
child_macronut['FAT'] = percent_to_kcalL(child_macronut['FAT'], 'fat')
child_macronut['CARBOHYDRATE'] = percent_to_kcalL(child_macronut['CARBOHYDRATE'], 'carb')

child_macronut.index = [remove_suffix(idx) for idx in child_macronut.index]
child_macronut=child_macronut.rename_axis("sample_ID")
vital_macronut=vital_macronut.rename_axis("sample_ID")
misame_macronut=misame_macronut.rename_axis("sample_ID")

In [10]:
table = "combined_table_with_ae.csv" 
targeted_metab_path = os.path.join(Dropbox_path, 'IMiC/Data/Targeted_metabolomics')
child_df = pd.read_csv(os.path.join(targeted_metab_path,'CHILD/Processed/'+table), index_col=0).rename_axis('sample_ID')
child_df.index = child_df.index.str.replace(r'-\d+$', '', regex=True)
vital_df = pd.read_csv(os.path.join(targeted_metab_path,'VITAL/Processed/'+table), index_col=0).rename_axis('sample_ID')
elicit_df = pd.read_csv(os.path.join(targeted_metab_path,'ELICIT/Processed/'+table), index_col=0).rename_axis('sample_ID')
misame_df = pd.read_csv(os.path.join(targeted_metab_path,'MISAME/Processed/'+table), index_col=0).rename_axis('sample_ID')

# Round 1:Full-feature analyses 

## Untargeted Analysis

In [11]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_1to1_preproc, how='right', on='sample_ID')
misame_BEP_meta_for_ML=pd.merge(misame_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(misame_BEP_meta_for_ML, 'postnatal_BEP')
misame_BEP_meta_gbm=run_gbm_bw(df=misame_BEP_meta_for_ML,
                                       outcome_col = 'postnatal_BEP',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':10000, 'max_depth':4},
                                       pickle_path='pickles/misame_BEP_meta_1to1_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
0.8831168831168831
ROC AUC mean:0.869814241486068
ROC AUC median:0.8614551083591331
F1 median:0.786628733997155
BER median:0.21336805555555555


In [12]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_1to1_preproc, how='right', on='sample_ID')
vital_BEP_meta_for_ML=pd.merge(vital_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(vital_BEP_meta_for_ML, 'postnatal_BEP')
vital_BEP_meta_gbm=run_gbm_bw(df=vital_BEP_meta_for_ML,
                                       outcome_col = 'postnatal_BEP',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':10000, 'max_depth':4},
                                       pickle_path='pickles/vital_BEP_meta_1to1_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
0.5
ROC AUC mean:0.8276666666666666
ROC AUC median:0.8166666666666667
F1 median:0.8235294117647058
BER median:0.277972027972028


In [13]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_1to1_preproc, how='right', on='sample_ID')
misame_WAZM04_meta_for_ML=pd.merge(misame_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(misame_WAZM04_meta_for_ML, 'WAZ_M04_wide_margin')
misame_WAZ_gbm_full=run_gbm_bw(df=misame_WAZM04_meta_for_ML.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':10000, 'max_depth':2},
                                       pickle_path='pickles/misame_WAZ_margin_full_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
1.9473684210526316
ROC AUC mean:0.6848214285714286
ROC AUC median:0.6766581632653061
F1 median:0.5266805266805267
BER median:0.39285714285714285


In [14]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_1to1_preproc, how='right', on='sample_ID')
vital_WAZM04_meta_for_ML=pd.merge(vital_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(vital_WAZM04_meta_for_ML, 'WAZ_M04_wide_margin')
vital_WAZ_gbm_full=run_gbm_bw(df=vital_WAZM04_meta_for_ML.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':10000, 'max_depth':2,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/vital_WAZ_margin_full_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
2.0
ROC AUC mean:0.615686274509804
ROC AUC median:0.6339869281045751
F1 median:0.30980392156862746
BER median:0.4624183006535948


In [11]:
child_waz_df = pd.merge(child_metadata[['SampleID','WAZ_M03_wide_margin']], child_1to1_preproc, left_on='SampleID', right_on='sample_ID').drop_duplicates().drop('sample_ID', axis=1).rename(columns={'SampleID':'SubjectID'})
child_waz_df = child_waz_df.rename(columns={'SampleID': 'SubjectID'})
outcome_ratio(child_waz_df, 'WAZ_M03_wide_margin')
child_WAZ_gbm_full=run_gbm_bw(df=child_waz_df.dropna(subset='WAZ_M03_wide_margin'),
                                       outcome_col = 'WAZ_M03_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':10000, 'max_depth':2,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/combined_child_waz_gap.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
1.7326732673267327
ROC AUC mean:0.5316363636363637
ROC AUC median:0.5436363636363637
F1 median:0.3100686498855835
BER median:0.4893181818181819


## Targeted Analysis

In [16]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_df, how='right', on='sample_ID')
misame_full_df_bep=pd.merge(misame_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna().drop_duplicates(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
misame_BEP_gbm_ae=run_gbm_bw(df=misame_full_df_bep,
                                       outcome_col = 'postnatal_BEP',
                                       seed_list=seedlist1,  
                                       confounder_list = [],
                                       params_dict = {'n_estimators':10000, 'max_depth':4},
                                       pickle_path='pickles/misame_BEP_uM_gbm_AE.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.8515082956259427
ROC AUC median:0.8540723981900453
F1 median:0.7875041764116271
BER median:0.22473604826546004


In [17]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_df, how='right', on='sample_ID')
vital_full_df_bep=pd.merge(vital_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna().drop_duplicates(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
vital_bep_gbm_ae=run_gbm_bw(df=vital_full_df_bep,
                                       outcome_col = 'postnatal_BEP',
                                       seed_list=seedlist1,                                        
                                       confounder_list = [],
                                       params_dict = {'n_estimators':10000, 'max_depth':4},
                                       pickle_path='pickles/vital_BEP_uM_gbm_AE.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.825230769230769
ROC AUC median:0.8384615384615384
F1 median:0.8148148148148148
BER median:0.32923076923076927


In [14]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_df, how='right', on='sample_ID')
misame_full_df_waz_margin=pd.merge(misame_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna().drop_duplicates(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(misame_full_df_waz_margin, 'WAZ_M04_wide_margin')
misame_wazmargin_gbm=run_gbm_bw(df=misame_full_df_waz_margin,
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':10000, 'max_depth':4,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/misame_wazmargin.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
1.9473684210526316
ROC AUC mean:0.659438775510204
ROC AUC median:0.6734693877551021
F1 median:0.4622222222222222
BER median:0.3660714285714286


In [13]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_df, how='right', on='sample_ID')
vital_full_df_waz_margin=pd.merge(vital_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna().drop_duplicates(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(vital_full_df_waz_margin, 'WAZ_M04_wide_margin')
vital_wazmargin_gbm=run_gbm_bw(df=vital_full_df_waz_margin,
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':10000, 'max_depth':4,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/vital_wazmargin.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
2.0
ROC AUC mean:0.5843137254901961
ROC AUC median:0.5588235294117647
F1 median:0.3875
BER median:0.43627450980392163


In [12]:
child_full_df_waz_margin=child_meta_for_ML[['SampleID', 'WAZ_M03_wide_margin']].merge(child_df, how='inner', left_on='SampleID', right_index=True)
child_full_df_waz_margin=child_full_df_waz_margin.rename(columns={'SampleID':'SubjectID'}).dropna(subset='WAZ_M03_wide_margin')
outcome_ratio(child_full_df_waz_margin, 'WAZ_M03_wide_margin')
child_wazmargin_gbm=run_gbm_bw(df=child_full_df_waz_margin,
                                       outcome_col = 'WAZ_M03_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':10000, 'max_depth':4,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/child_wazmargin.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
1.7272727272727273
ROC AUC mean:0.5553488372093024
ROC AUC median:0.5576744186046512
F1 median:0.3904761904761905
BER median:0.4537209302325581


# Feature Extraction

## Untargeted

In [11]:
misame_BEP_meta_gbm = pd.read_pickle('pickles/misame_BEP_meta_1to1_gbm.pickle')
vital_BEP_meta_1to1_gbm = pd.read_pickle('pickles/vital_BEP_meta_1to1_gbm.pickle')

Misame_BEP_feature_max_df,Misame_BEP_feature_mean_df = get_top_feats(misame_BEP_meta_gbm)
Misame_top_BEP_1_50_max=rank_and_filter(df=Misame_BEP_feature_max_df.sort_values(by='Max_Max_Abs', ascending=False),
                                        seed_cutoff=1, rank_cutoff=50)
Misame_top_BEP_1_50_mean=rank_and_filter(df=Misame_BEP_feature_mean_df.sort_values(by='Max_Max_Abs', ascending=False),
                                         seed_cutoff=1, rank_cutoff=50)   

Vital_BEP_feature_max_df,Vital_BEP_feature_mean_df = get_top_feats(vital_BEP_meta_1to1_gbm)
Vital_top_BEP_1_50_max=rank_and_filter(df=Vital_BEP_feature_max_df.sort_values(by='Max_Max_Abs', ascending=False),
                                        seed_cutoff=1, rank_cutoff=50)
Vital_top_BEP_1_50_mean=rank_and_filter(df=Vital_BEP_feature_mean_df.sort_values(by='Max_Max_Abs', ascending=False),
                                         seed_cutoff=1, rank_cutoff=50)   

cross_study_BEP_1_50_max=list(set(Misame_top_BEP_1_50_max['Feature']) & set(Vital_top_BEP_1_50_max['Feature']))
cross_study_BEP_1_50_mean=list(set(Misame_top_BEP_1_50_mean['Feature']) & set(Vital_top_BEP_1_50_mean['Feature']))
mean_max_union_set_BEP=list(set(cross_study_BEP_1_50_mean) | set(cross_study_BEP_1_50_max))
print(len(mean_max_union_set_BEP))

(384, 14)
(315, 14)
(405, 14)
(397, 14)
21


In [12]:
vital_WAZ_gbm_full=pd.read_pickle("../pickles/vital_WAZ_margin_full_gbm.pickle")
misame_WAZ_gbm_full=pd.read_pickle("../pickles/combined_misame_waz_gap.pickle")
child_WAZ_gbm_full=pd.read_pickle("../pickles/combined_child_waz_gap.pickle")

Vital_WAZ_feature_max_df,Vital_WAZ_feature_mean_df = get_top_feats(vital_WAZ_gbm_full)
Vital_top_WAZ_1_50_max=rank_and_filter(df=Vital_WAZ_feature_max_df.sort_values(by='Max_Max_Abs', ascending=False),
                                        seed_cutoff=1, rank_cutoff=50)
Vital_top_WAZ_1_50_mean=rank_and_filter(df=Vital_WAZ_feature_mean_df.sort_values(by='Max_Max_Abs', ascending=False),
                                         seed_cutoff=1, rank_cutoff=50)
vital_margin_feats_400 = set(Vital_top_WAZ_1_50_max.iloc[:400]['Feature']) | set(Vital_top_WAZ_1_50_mean.iloc[:400]['Feature'])

Misame_WAZ_feature_max_df,Misame_WAZ_feature_mean_df = get_top_feats(misame_WAZ_gbm_full)
Misame_top_WAZ_1_50_max=rank_and_filter(df=Misame_WAZ_feature_max_df.sort_values(by='Max_Max_Abs', ascending=False),
                                        seed_cutoff=1, rank_cutoff=50)
Misame_top_WAZ_1_50_mean=rank_and_filter(df=Misame_WAZ_feature_mean_df.sort_values(by='Max_Max_Abs', ascending=False),
                                         seed_cutoff=1, rank_cutoff=50)
misame_margin_feats_400= set(Misame_top_WAZ_1_50_max.iloc[:400]['Feature']) | set(Misame_top_WAZ_1_50_mean.iloc[:400]['Feature'])

Child_WAZ_feature_max_df,Child_WAZ_feature_mean_df = get_top_feats(child_WAZ_gbm_full)
Child_top_WAZ_1_50_max=rank_and_filter(df=Child_WAZ_feature_max_df.sort_values(by='Max_Max_Abs', ascending=False),
                                        seed_cutoff=1, rank_cutoff=50)
Child_top_WAZ_1_50_mean=rank_and_filter(df=Child_WAZ_feature_mean_df.sort_values(by='Max_Max_Abs', ascending=False),
                                         seed_cutoff=1, rank_cutoff=50)
child_margin_feats_400 = set(Child_top_WAZ_1_50_max.iloc[:350]['Feature']) | set(Child_top_WAZ_1_50_mean.iloc[:350]['Feature'])

child_misame_feats = list(set(child_margin_feats_400).intersection(set(misame_margin_feats_400)))
child_vital_feats = list(set(child_margin_feats_400).intersection(set(vital_margin_feats_400)))
misame_vital_feats = list(set(misame_margin_feats_400).intersection(set(vital_margin_feats_400)))
combined_waz_feats = list(set(child_misame_feats+child_vital_feats+misame_vital_feats))
print(len(combined_waz_feats))

(411, 14)
(409, 14)
(427, 14)
(421, 14)
(390, 14)
(350, 14)
29


## Targeted

In [13]:
# TARGETED BEP

rank_cutoff = 60 
seed_cutoff = 3 

misame_BEP_gbm_ae = pd.read_pickle('pickles/misame_BEP_uM_gbm_AE.pickle')
vital_bep_gbm_ae = pd.read_pickle('pickles/vital_BEP_uM_gbm_AE.pickle')

# Extracting top features based on max shap, mean shap
misame_BEP_uM_max, misame_BEP_uM_mean = get_top_feats(misame_BEP_gbm_ae)
vital_BEP_uM_max, vital_BEP_uM_mean = get_top_feats(vital_bep_gbm_ae)

vital_BEP_uM_max_top = rank_and_filter(vital_BEP_uM_max, seed_cutoff, rank_cutoff)['Feature'].to_list()
vital_BEP_uM_mean_top = rank_and_filter(vital_BEP_uM_mean, seed_cutoff, rank_cutoff)['Feature'].to_list()

misame_BEP_uM_max_top = rank_and_filter(misame_BEP_uM_max, seed_cutoff, rank_cutoff)['Feature'].to_list()
misame_BEP_uM_mean_top = rank_and_filter(misame_BEP_uM_mean, seed_cutoff, rank_cutoff)['Feature'].to_list()

# Identifying union between sets of top shap values based on max and mean
max_bep_int = set(vital_BEP_uM_max_top).intersection(set(misame_BEP_uM_max_top))
mean_bep_int = set(vital_BEP_uM_mean_top).intersection(set(misame_BEP_uM_mean_top))
combined_bep_int = list(set( list(max_bep_int)+(list(mean_bep_int) )))
print(len(combined_bep_int))

# Saving extracted features
if not os.path.exists('cross_study_features'):
    os.makedirs('cross_study_features')
with open('cross_study_features/targeted_crossBEP.txt', 'w') as f:
    f.write('\n'.join(combined_bep_int))

(70, 14)
(71, 14)
(65, 14)
(62, 14)
20


In [14]:
# TARGETED WAZ

rank_cutoff = 40 
seed_cutoff = 3

misame_wazmargin_gbm = pd.read_pickle('pickles/misame_wazmargin.pickle')
vital_wazmargin_gbm = pd.read_pickle('pickles/vital_wazmargin.pickle')
child_wazmargin_gbm = pd.read_pickle('pickles/child_wazmargin.pickle')


vital_WAZ_uM_max, vital_WAZ_uM_mean = get_top_feats(vital_wazmargin_gbm)
misame_WAZ_uM_max, misame_WAZ_uM_mean = get_top_feats(misame_wazmargin_gbm)
child_WAZ_uM_max, child_WAZ_uM_mean = get_top_feats(child_wazmargin_gbm)

vital_WAZ_uM_max_top = rank_and_filter(vital_WAZ_uM_max, seed_cutoff, rank_cutoff)['Feature'].to_list()
vital_WAZ_uM_mean_top = rank_and_filter(vital_WAZ_uM_mean, seed_cutoff, rank_cutoff)['Feature'].to_list()

misame_WAZ_uM_max_top = rank_and_filter(misame_WAZ_uM_max, seed_cutoff, rank_cutoff)['Feature'].to_list()
misame_WAZ_uM_mean_top = rank_and_filter(misame_WAZ_uM_mean, seed_cutoff, rank_cutoff)['Feature'].to_list()

child_WAZ_uM_max_top = rank_and_filter(child_WAZ_uM_max, seed_cutoff, rank_cutoff)['Feature'].to_list()
child_WAZ_uM_mean_top = rank_and_filter(child_WAZ_uM_mean, seed_cutoff, rank_cutoff)['Feature'].to_list()

child_misame_max_waz = set(child_WAZ_uM_max_top).intersection(set(misame_WAZ_uM_max_top))
child_vital_max_waz = set(child_WAZ_uM_max_top).intersection(set(vital_WAZ_uM_max_top))
misame_vital_max_waz = set(misame_WAZ_uM_max_top).intersection(set(vital_WAZ_uM_max_top))

child_misame_mean_waz = set(child_WAZ_uM_mean_top).intersection(set(misame_WAZ_uM_mean_top))
child_vital_mean_waz = set(child_WAZ_uM_mean_top).intersection(set(vital_WAZ_uM_mean_top))
misame_vital_mean_waz = set(misame_WAZ_uM_mean_top).intersection(set(vital_WAZ_uM_mean_top))

combined_waz_int = list(set( list(child_misame_max_waz)+(list(child_vital_max_waz)+list(misame_vital_max_waz)+list(child_misame_mean_waz)+list(child_vital_mean_waz)+list(misame_vital_mean_waz) )))
print(len(combined_waz_int))

# Saving extracted features
if not os.path.exists('cross_study_features'):
    os.makedirs('cross_study_features')
with open('cross_study_features/targeted_crossWAZ.txt', 'w') as f:
    f.write('\n'.join(combined_waz_int))

(37, 14)
(41, 14)
(47, 14)
(49, 14)
(52, 14)
(58, 14)
29


# Round 2:Analysis with top features 

## Untargeted analysis

In [ ]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            #misame_preproc, how='right', on='sample_ID')
                            misame_1to1_preproc[['sample_ID'] + mean_max_union_set_BEP], how='right', on='sample_ID')
misame_BEP_fromMeanMaxUnionBEP_meta_for_ML=pd.merge(misame_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(misame_BEP_fromMeanMaxUnionBEP_meta_for_ML, 'postnatal_BEP')
misame_BEP_fromMeanMaxUnionBEP_meta_for_ML_gbm=run_gbm_bw(df=misame_BEP_fromMeanMaxUnionBEP_meta_for_ML,
                                       outcome_col = 'postnatal_BEP',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                                     params_dict = {'n_estimators':10000, 'max_depth':4},
                                       pickle_path='pickles/misame_BEP_fromMeanMaxUnionBEP_meta_for_ML_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
0.8831168831168831
ROC AUC mean:0.9107088989441932
ROC AUC median:0.9159125188536954
F1 median:0.8408057179987004
BER median:0.17062594268476625


In [ ]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            #vital_preproc, how='right', on='sample_ID')
                            vital_1to1_preproc[['sample_ID'] + mean_max_union_set_BEP], how='right', on='sample_ID')
vital_BEP_fromMeanMaxUnionBEP_meta_for_ML=pd.merge(vital_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(vital_BEP_fromMeanMaxUnionBEP_meta_for_ML, 'postnatal_BEP')
vital_BEP_fromMeanMaxUnionBEP_meta_for_ML_gbm=run_gbm_bw(df=vital_BEP_fromMeanMaxUnionBEP_meta_for_ML,
                                       outcome_col = 'postnatal_BEP',
                                       confounder_list = [],
                                       seed_list=seedlist2, 
                                                     params_dict = {'n_estimators':10000, 'max_depth':4},
                                       pickle_path='pickles/vital_BEP_fromMeanMaxUnionBEP_meta_for_ML_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
0.5
ROC AUC mean:0.8381538461538461
ROC AUC median:0.8553846153846154
F1 median:0.8268590455049944
BER median:0.2530769230769231


In [ ]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            #misame_preproc, how='right', on='sample_ID')
                            misame_1to1_preproc[['sample_ID'] + combined_waz_feats], how='right', on='sample_ID')
misame_waz_cs_overlap=pd.merge(misame_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(misame_waz_cs_overlap, 'WAZ_M04_wide_margin')
misame_cs_overlap_feats = run_gbm_bw(df=misame_waz_cs_overlap.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/misame_cs_overlap_sig_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
1.9473684210526316
ROC AUC mean:0.6543367346938775
ROC AUC median:0.6823979591836735
F1 median:0.4542857142857143
BER median:0.4017857142857143


In [ ]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            #vital_preproc, how='right', on='sample_ID')
                            vital_1to1_preproc[['sample_ID'] + combined_waz_feats], how='right', on='sample_ID')
vital_waz_cs_overlap=pd.merge(vital_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
outcome_ratio(vital_waz_cs_overlap, 'WAZ_M04_wide_margin')
vital_cs_overlap_feats = run_gbm_bw(df=vital_waz_cs_overlap.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/vital_waz_cs_overlap_sig_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

Ratio outcome 0 / outcome 1:
2.0
ROC AUC mean:0.7915032679738562
ROC AUC median:0.8071895424836601
F1 median:0.5941176470588235
BER median:0.3088235294117647


In [ ]:
child_waz_df = pd.merge(child_metadata[['SampleID','WAZ_M03_wide_margin']], child_1to1_preproc, left_on='SampleID', right_on='sample_ID').drop_duplicates().drop('sample_ID', axis=1).rename(columns={'SampleID':'SubjectID'})
child_waz_df = child_waz_df.rename(columns={'SampleID': 'SubjectID'})
child_waz_csoverlap = child_waz_df[['SubjectID','WAZ_M03_wide_margin']+ [col for col in child_waz_df.columns if any(feat in col for feat in combined_waz_feats)]]
child_WAZ_cs_overlap = run_gbm_bw(df=child_waz_csoverlap.dropna(subset='WAZ_M03_wide_margin'),
                                       outcome_col = 'WAZ_M03_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/child_WAZ_cs_overlap_sig_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.7365454545454545
ROC AUC median:0.7277272727272728
F1 median:0.5714285714285714
BER median:0.3322727272727273


## Targeted analysis

In [28]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_df, how='right', on='sample_ID')
misame_full_df_bep=pd.merge(misame_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna().drop_duplicates(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
columns_to_keep = [col for col in misame_full_df_bep.columns if col[4:] in combined_bep_int]
misame_bep_top_gbm_ae=run_gbm_bw(df=misame_full_df_bep[['SubjectID','postnatal_BEP']+columns_to_keep],
                                       outcome_col = 'postnatal_BEP',
                                       seed_list=seedlist1,                                        
                                       confounder_list = [],
                                       params_dict = {'n_estimators':100, 'max_depth':2},
                                       pickle_path='pickles/misame_BEP_topfeats_ae.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.8630467571644042
ROC AUC median:0.8725490196078431
F1 median:0.8001250195343022
BER median:0.21191553544494723


In [29]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_df, how='right', on='sample_ID')
vital_full_df_bep=pd.merge(vital_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna().drop_duplicates(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
columns_to_keep = [col for col in vital_full_df_bep.columns if col[4:] in combined_bep_int]
vital_bep_top_gbm_ae=run_gbm_bw(df=vital_full_df_bep[['SubjectID','postnatal_BEP']+columns_to_keep],
                                       outcome_col = 'postnatal_BEP',
                                       seed_list=seedlist1,                                        
                                       confounder_list = [],
                                       params_dict = {'n_estimators':100, 'max_depth':2},
                                       pickle_path='pickles/vital_BEP_topfeats_ae.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.8643076923076924
ROC AUC median:0.8676923076923078
F1 median:0.8653348131705513
BER median:0.23230769230769233


In [30]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            #misame_preproc, how='right', on='sample_ID')
                            misame_df, how='right', on='sample_ID')
misame_full_df_waz_margin=pd.merge(misame_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna().drop_duplicates(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
columns_to_keep = [col for col in misame_full_df_waz_margin.columns if col[4:] in combined_waz_int]
misame_wazmargin_crosswaz_gbm=run_gbm_bw(df=misame_full_df_waz_margin[['SubjectID','WAZ_M04_wide_margin']+columns_to_keep],
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':1000, 'max_depth':2,'alpha':0.2,'scale_pos_weight':2},
                                       pickle_path='pickles/misame_wazmargin_crosswaz_ae_pure.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.6775510204081632
ROC AUC median:0.6709183673469388
F1 median:0.4481605351170569
BER median:0.39285714285714285


In [31]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_df, how='right', on='sample_ID')
vital_full_df_waz_margin=pd.merge(vital_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna().drop_duplicates(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
columns_to_keep = [col for col in vital_full_df_waz_margin.columns if col[4:] in combined_waz_int]
vital_wazmargin_crosswaz_gbm=run_gbm_bw(df=vital_full_df_waz_margin[['SubjectID','WAZ_M04_wide_margin']+columns_to_keep],
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2,'alpha':0.5, 'scale_pos_weight':2},
                                       pickle_path='pickles/vital_wazmargin_crosswaz_ae_pure.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.6529411764705882
ROC AUC median:0.6764705882352942
F1 median:0.424812030075188
BER median:0.4199346405228759


In [32]:
child_full_df_waz_margin=child_meta_for_ML[['SampleID', 'WAZ_M03_wide_margin']].merge(child_df, how='inner', left_on='SampleID', right_index=True)
child_full_df_waz_margin=child_full_df_waz_margin.rename(columns={'SampleID':'SubjectID'})
child_wazmargin_crosswaz_gbm=run_gbm_bw(df=child_full_df_waz_margin[['SubjectID','WAZ_M03_wide_margin']+combined_waz_int].dropna(),
                                       outcome_col = 'WAZ_M03_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2, 'scale_pos_weight':2},
                                       pickle_path='pickles/child_wazmargin_crosswaz_ae_pure.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.6402790697674419
ROC AUC median:0.6427906976744187
F1 median:0.4753191489361702
BER median:0.4095348837209302


# MS2 Results

## 104 Untargeted Features sent for MS2 Annotation

In [ ]:
annotated_feats_df = pd.read_excel(os.path.join(Dropbox_path, 'IMiC/Code/machine_learning/IMiC_Azad_Metabolite_ID_Share.xlsx'))
feat104 = annotated_feats_df['mtb_id_MISAME3'].to_list()

In [ ]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_1to1_preproc, how='right', on='sample_ID')
misame_WAZM04_meta_for_ML=pd.merge(misame_full_BM_df_filt[['SubjectID', 'postnatal_BEP']].drop_duplicates(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
misame_bep_sig = misame_WAZM04_meta_for_ML[['SubjectID','postnatal_BEP']+ [col for col in misame_WAZM04_meta_for_ML.columns if any(feat in col for feat in feat104)]]
misame_waz_all_gbm = run_gbm_bw(df=misame_bep_sig.dropna(subset='postnatal_BEP'),
                                       outcome_col = 'postnatal_BEP',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2},
                                       pickle_path='pickles/misame_bep_all_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.8980392156862745
ROC AUC median:0.8981900452488688
F1 median:0.8353375020041687
BER median:0.17609351432880846


In [ ]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_1to1_preproc, how='right', on='sample_ID')
vital_WAZM04_meta_for_ML=pd.merge(vital_full_BM_df_filt[['SubjectID','postnatal_BEP']].drop_duplicates(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
vital_bep_sig = vital_WAZM04_meta_for_ML[['SubjectID','postnatal_BEP']+ [col for col in vital_WAZM04_meta_for_ML.columns if any(feat in col for feat in feat104)]]
vital_waz_all_gbm = run_gbm_bw(df=vital_bep_sig.dropna(subset='postnatal_BEP'),
                                       outcome_col = 'postnatal_BEP',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2, 'reg_alpha':5},
                                       pickle_path='pickles/vital_bep_all_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.8178461538461539
ROC AUC median:0.8023076923076923
F1 median:0.8076923076923077
BER median:0.32846153846153847


In [ ]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_1to1_preproc, how='right', on='sample_ID')
misame_WAZM04_meta_for_ML=pd.merge(misame_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
misame_waz_all = misame_WAZM04_meta_for_ML[['SubjectID','WAZ_M04_wide_margin']+ [col for col in misame_WAZM04_meta_for_ML.columns if any(feat in col for feat in feat104)]]
misame_waz_all_gbm = run_gbm_bw(df=misame_waz_all.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/misame_all_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.7293367346938775
ROC AUC median:0.7193877551020409
F1 median:0.5558620689655173
BER median:0.3392857142857143


In [ ]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_1to1_preproc, how='right', on='sample_ID')
vital_WAZM04_meta_for_ML=pd.merge(vital_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
vital_waz_all = vital_WAZM04_meta_for_ML[['SubjectID','WAZ_M04_wide_margin']+ [col for col in vital_WAZM04_meta_for_ML.columns if any(feat in col for feat in feat104)]]
vital_WAZ_all_gbm= run_gbm_bw(df=vital_waz_all.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1, 'scale_pos_weight':2},
                                       pickle_path='pickles/vital_waz_all_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.8601307189542483
ROC AUC median:0.8790849673202614
F1 median:0.6625
BER median:0.2549019607843137


In [ ]:
child_waz_df = pd.merge(child_metadata[['SampleID','WAZ_M03_wide_margin']], child_1to1_preproc, left_on='SampleID', right_on='sample_ID').drop_duplicates().drop('sample_ID', axis=1).rename(columns={'SampleID':'SubjectID'})
child_waz_df = child_waz_df.rename(columns={'SampleID': 'SubjectID'})
child_waz_all = child_waz_df[['SubjectID','WAZ_M03_wide_margin']+ [col for col in child_waz_df.columns if any(feat in col for feat in feat104)]]
child_WAZ_all_gbm = run_gbm_bw(df=child_waz_all.dropna(subset='WAZ_M03_wide_margin'),
                                       outcome_col = 'WAZ_M03_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/child_WAZ_all_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.858
ROC AUC median:0.86
F1 median:0.6957759848125297
BER median:0.23681818181818182


## Comparing Annotated vs Unannotated MS2 Features

In [ ]:
annotated_feats_df = pd.read_excel(os.path.join(Dropbox_path, 'IMiC/Code/machine_learning/IMiC_Azad_Metabolite_ID_Share.xlsx'))
annotated_feats = annotated_feats_df[(~annotated_feats_df['Correlation_Cluster'].isna()) | (~annotated_feats_df['Compound_ID'].isna())]['mtb_id_MISAME3'].tolist()
not_annotated_feats = annotated_feats_df['mtb_id_MISAME3'].tolist()
not_annotated_feats = [item for item in not_annotated_feats if item not in annotated_feats]

In [ ]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_1to1_preproc, how='right', on='sample_ID')
misame_WAZM04_meta_for_ML=pd.merge(misame_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
misame_waz_annotated = misame_WAZM04_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']+ [col for col in misame_WAZM04_meta_for_ML.columns if any(feat in col for feat in annotated_feats)]]
misame_waz_annotated_gbm_full=run_gbm_bw(df=misame_waz_annotated.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2,'gamma':5},
                                       pickle_path='pickles/misame_waz_annotated_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.6483418367346939
ROC AUC median:0.6447704081632653
F1 median:0.5333333333333333
BER median:0.3571428571428571


In [ ]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_1to1_preproc, how='right', on='sample_ID')
vital_WAZM04_meta_for_ML=pd.merge(vital_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
vital_waz_annotated = vital_WAZM04_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']+ [col for col in vital_WAZM04_meta_for_ML.columns if any(feat in col for feat in annotated_feats)]]
vital_waz_annotated_gbm_full=run_gbm_bw(df=vital_waz_annotated.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2,'gamma':5},
                                       pickle_path='pickles/vital_waz_annotated_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.6787581699346406
ROC AUC median:0.6666666666666667
F1 median:0.513157894736842
BER median:0.3562091503267974


In [ ]:
child_waz_df = pd.merge(child_metadata[['SampleID','WAZ_M03_wide_margin']], child_1to1_preproc, left_on='SampleID', right_on='sample_ID').drop_duplicates().drop('sample_ID', axis=1).rename(columns={'SampleID':'SubjectID'})
child_waz_df = child_waz_df.rename(columns={'SampleID': 'SubjectID'})
child_waz_annotated = child_waz_df[['SubjectID', 'WAZ_M03_wide_margin']+ [col for col in child_waz_df.columns if any(feat in col for feat in annotated_feats)]]
child_waz_annotated_gbm_full=run_gbm_bw(df=child_waz_annotated.dropna(subset='WAZ_M03_wide_margin'),
                                       outcome_col = 'WAZ_M03_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2,'gamma':5},
                                       pickle_path='pickles/child_waz_annotated_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.6283181818181818
ROC AUC median:0.605909090909091
F1 median:0.5373514431239388
BER median:0.39022727272727276


In [ ]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_1to1_preproc, how='right', on='sample_ID')
misame_WAZM04_meta_for_ML=pd.merge(misame_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(misame_preproc_hits), how='inner', on='SubjectID')
misame_waz_notannotated = misame_WAZM04_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']+ [col for col in misame_WAZM04_meta_for_ML.columns if any(feat in col for feat in not_annotated_feats)]]
misame_waz_not_annotated_gbm_full=run_gbm_bw(df=misame_waz_notannotated.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2,'gamma':5},
                                       pickle_path='pickles/misame_waz_not_annotated_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.7122448979591838
ROC AUC median:0.7079081632653061
F1 median:0.4813793103448276
BER median:0.3839285714285714


In [ ]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_1to1_preproc, how='right', on='sample_ID')
vital_WAZM04_meta_for_ML=pd.merge(vital_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna(), timepoint_col_distribute(vital_preproc_hits), how='inner', on='SubjectID')
vital_waz_notannotated = vital_WAZM04_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']+ [col for col in vital_WAZM04_meta_for_ML.columns if any(feat in col for feat in not_annotated_feats)]]
vital_waz_not_annotated_gbm_full=run_gbm_bw(df=vital_waz_notannotated.dropna(subset='WAZ_M04_wide_margin'),
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2,'gamma':5},
                                       pickle_path='pickles/vital_waz_not_annotated_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.7836601307189542
ROC AUC median:0.8235294117647058
F1 median:0.5444444444444445
BER median:0.3382352941176471


In [ ]:
child_waz_df = pd.merge(child_metadata[['SampleID','WAZ_M03_wide_margin']], child_1to1_preproc, left_on='SampleID', right_on='sample_ID').drop_duplicates().drop('sample_ID', axis=1).rename(columns={'SampleID':'SubjectID'})
child_waz_df = child_waz_df.rename(columns={'SampleID': 'SubjectID'})
child_waz_not_annotated = child_waz_df[['SubjectID', 'WAZ_M03_wide_margin']+ [col for col in child_waz_df.columns if any(feat in col for feat in not_annotated_feats)]]
child_waz_not_annotated_gbm_full=run_gbm_bw(df=child_waz_not_annotated.dropna(subset='WAZ_M03_wide_margin'),
                                       outcome_col = 'WAZ_M03_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':1,'max_delta_step':5, 'scale_pos_weight':2,'gamma':5},
                                       pickle_path='pickles/child_waz_not_annotated_feats_gbm.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.8401818181818183
ROC AUC median:0.8354545454545454
F1 median:0.6973063973063973
BER median:0.23931818181818182


# Baseline models

In [40]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_df, how='right', on='sample_ID')
misame_full_df_bep=misame_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna().drop_duplicates()
misame_full_df_bep['Random_Int'] = np.random.randint(0, 1000, size=len(misame_full_df_bep)) # Random integer for baseline model
misame_bep_baseline=run_gbm_bw(df=misame_full_df_bep[['SubjectID','postnatal_BEP', 'Random_Int']],
                                       outcome_col = 'postnatal_BEP',
                                       seed_list=seedlist1,                                        
                                       confounder_list = [],
                                       params_dict = {'n_estimators':100, 'max_depth':2},
                                       pickle_path='pickles/misame_BEP_baseline.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.5540346907993967
ROC AUC median:0.5369532428355958
F1 median:0.5676376391642519
BER median:0.4596530920060332


In [41]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_df, how='right', on='sample_ID')
vital_full_df_bep=vital_meta_for_ML[['SubjectID', 'postnatal_BEP']].dropna().drop_duplicates()
vital_full_df_bep['Random_Int'] = np.random.randint(0, 1000, size=len(vital_full_df_bep)) # Random integer for baseline model
vital_bep_baseline=run_gbm_bw(df=vital_full_df_bep[['SubjectID','postnatal_BEP', 'Random_Int']],
                                       outcome_col = 'postnatal_BEP',
                                       seed_list=seedlist1,                                        
                                       confounder_list = [],
                                       params_dict = {'n_estimators':100, 'max_depth':2},
                                       pickle_path='pickles/vital_BEP_baseline.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.4715384615384616
ROC AUC median:0.4430769230769231
F1 median:0.7207792207792207
BER median:0.5038461538461538


In [42]:
misame_preproc_hits=pd.merge(misame_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            misame_df, how='right', on='sample_ID')
misame_full_df_waz_margin=misame_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna().drop_duplicates()
misame_full_df_waz_margin['Random_Int'] = np.random.randint(0, 1000, size=len(misame_full_df_waz_margin)) # Random integer for baseline model
misame_wazmargin_baseline_gbm=run_gbm_bw(df=misame_full_df_waz_margin[['SubjectID','WAZ_M04_wide_margin', 'Random_Int']],
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2, 'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/misame_wazmargin_baseline.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.4764030612244897
ROC AUC median:0.4464285714285714
F1 median:0.3693957115009746
BER median:0.49107142857142855


In [43]:
vital_preproc_hits=pd.merge(vital_full_BM_df_filt.rename(columns={'SampleID': 'sample_ID'}, inplace=False)[['sample_ID', 'SubjectID', 'Timepoint']], 
                            vital_df, how='right', on='sample_ID')
vital_full_df_waz_margin=vital_meta_for_ML[['SubjectID', 'WAZ_M04_wide_margin']].dropna().drop_duplicates()
vital_full_df_waz_margin['Random_Int'] = np.random.randint(0, 1000, size=len(vital_full_df_waz_margin)) # Random integer for baseline model
vital_wazmargin_baseline_gbm=run_gbm_bw(df=vital_full_df_waz_margin[['SubjectID','WAZ_M04_wide_margin', 'Random_Int']],
                                       outcome_col = 'WAZ_M04_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2, 'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/vital_wazmargin_baseline.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.39281045751633986
ROC AUC median:0.37254901960784315
F1 median:0.2833333333333333
BER median:0.542483660130719


In [44]:
child_waz_df = pd.merge(child_metadata[['SampleID','WAZ_M03_wide_margin']], child_1to1_preproc, left_on='SampleID', right_on='sample_ID').drop_duplicates().drop('sample_ID', axis=1).rename(columns={'SampleID':'SubjectID'})
child_waz_df = child_waz_df.rename(columns={'SampleID': 'SubjectID'})
child_waz_baseline_df=child_waz_df[['SubjectID', 'WAZ_M03_wide_margin']].dropna().drop_duplicates()
child_waz_baseline_df['Random_Int'] = np.random.randint(0, 1000, size=len(child_waz_baseline_df)) # Random integer for baseline model
child_waz_not_annotated_gbm_full=run_gbm_bw(df=child_waz_baseline_df.dropna(subset='WAZ_M03_wide_margin'),
                                       outcome_col = 'WAZ_M03_wide_margin',
                                       confounder_list = [],
                                       seed_list=seedlist1, 
                                       params_dict = {'n_estimators':100, 'max_depth':2,'max_delta_step':5, 'scale_pos_weight':2},
                                       pickle_path='pickles/child_wazmargin_baseline.pickle',
                                       use_SMOTE=False,
                                       startover=False) 

ROC AUC mean:0.5610909090909091
ROC AUC median:0.5511363636363636
F1 median:0.4869933454325469
BER median:0.4343181818181818


# Statsistics export

In [11]:
from sklearn.metrics import (roc_curve, auc, confusion_matrix, f1_score, precision_recall_curve, balanced_accuracy_score, precision_score, recall_score, average_precision_score)

def calc_ML_stats_from_file(pickle_file): 
    # read from file
    pickle = pd.read_pickle(pickle_file)
    roc_scores = []
    f1_scores = []
    ber_scores = []
    precision_scores = []
    recall_scores = []
    balanced_accs = []
    ap_scores = []
    prevalence_scores = []
    for i in pickle.keys():
        Fit_model = pickle[i]['Fit_model']
        X_testset = pickle[i]['X_testset']
        y_testset = pickle[i]['y_testset']
        y_true = y_testset
        y_pred = Fit_model.predict(X_testset)
        y_pred_prob = Fit_model.predict_proba(X_testset)[:, 1]
        # Calculate AUC ROC
        fpr, tpr, thresholds = roc_curve(y_testset, y_pred_prob, pos_label=1)
        roc_auc = auc(fpr, tpr)
        roc_scores.append(roc_auc)
        # Calculate F1
        f1=f1_score(y_testset, y_pred)
        f1_scores.append(f1)
        # Calculate Balanced ACC
        balanced_acc = balanced_accuracy_score(y_true, y_pred)
        balanced_accs.append(balanced_acc)
        # Calculate BER
        ber = 1 - balanced_acc
        ber_scores.append(ber)
        # Calculate precision 
        prec = precision_score(y_testset, y_pred)
        precision_scores.append(prec)
        # Calculate recall 
        rec = recall_score(y_testset, y_pred)
        recall_scores.append(rec)
        # Calculate Average Precision (area under PRC)
        ap = average_precision_score(y_testset, y_pred_prob)
        ap_scores.append(ap)
        # Calculate Prevalence
        prevalence = y_testset.sum() / len(y_testset)
        prevalence_scores.append(prevalence)

    df = pd.DataFrame({
        'Seed': pickle.keys(), 
        'F1': f1_scores, 
        'AUC': roc_scores, 
        'BER': ber_scores, 
        'Balanced_acc': balanced_accs, 
        'Precision': precision_scores, 
        'Recall': recall_scores,
        'AP': ap_scores,
        'Prevalence': prevalence_scores,
        'Run': os.path.basename(pickle_file)
    })
    return(df)

In [12]:
# Export stats 
temp_stats_file = os.path.join(Dropbox_path, "IMiC/Code/machine_learning/ML_stats_output.temp.csv")
final_stats_file = os.path.join(Dropbox_path, "IMiC/Code/machine_learning/ML_stats_output.csv")

stats_df_list = []
for pickle_file in os.listdir('pickles/'):
    pickle_path = os.path.join('pickles/', pickle_file)
    stats_df_list.append(calc_ML_stats_from_file(pickle_path))
    stats_report = pd.concat(stats_df_list, ignore_index=True)
    stats_report.to_csv(temp_stats_file)
stats_report = pd.concat(stats_df_list, ignore_index=True)
stats_report.to_csv(final_stats_file)